# LeetCode 1214 — Two Sum BSTs bằng Python

Link đề: https://leetcode.com/problems/two-sum-bsts/ · Độ khó: **Medium** · Chủ đề: Two Pointers, Binary Search, Stack, Tree, Depth-First Search, Binary Search Tree, Binary Tree

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề và chữ ký hàm trong notebook được đối chiếu với bản mô tả công khai tại https://leetcode.doocs.org/en/lc/1214/ (tên method `twoSumBSTs`, tham số `root1, root2, target`).

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def twoSumBSTs(self, root1: Optional[TreeNode], root2: Optional[TreeNode], target: int) -> bool:
        ...
```

Ghi chú: đáp án chỉ là `True`/`False` — có hay không **một nút ở cây 1** và **một nút ở cây 2** có tổng giá trị bằng `target`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **cây nhị phân tìm kiếm** (Binary Search Tree — BST): mọi giá trị ở cây con trái nhỏ hơn nút, mọi giá trị ở cây con phải lớn hơn nút; và hệ quả quan trọng nhất: **duyệt trung thứ tự (inorder) cho dãy tăng dần**.
2. Biết đưa một bài "trên cây" về bài quen thuộc **Two Sum** (bài 1) bằng cách duyệt cây thành danh sách hoặc tập hợp.
3. Đi từ **vét cạn O(n·m)** → **tập băm (hash set) O(n+m)** → **tìm kiếm nhị phân trên BST** → **hai con trỏ trên hai dãy đã sắp xếp** → **hai iterator dùng stack, bộ nhớ O(h)**.
4. Tự cài đặt duyệt cây **không đệ quy** (dùng stack) để tránh lỗi tràn đệ quy với cây lệch 5000 nút.
5. Tránh các lỗi kinh điển: lấy cả hai số trong cùng một cây, hai con trỏ cùng chạy một chiều, đi sai hướng khi tìm trong BST.

## 2. Đề bài & ví dụ

Cho gốc của **hai cây nhị phân tìm kiếm** `root1` và `root2`, cùng một số nguyên `target`. Trả về `True` khi và chỉ khi tồn tại **một nút trong cây thứ nhất** và **một nút trong cây thứ hai** sao cho tổng giá trị của chúng bằng `target`.

Lưu ý: hai số **phải đến từ hai cây khác nhau** — không được lấy hai nút trong cùng một cây.

**Ví dụ 1:** `root1 = [2,1,4]`, `root2 = [1,0,3]`, `target = 5` → `True` (vì `2 + 3 = 5`).

**Ví dụ 2:** `root1 = [0,-10,10]`, `root2 = [5,1,7,0,2]`, `target = 18` → `False`.

Cây được cho theo **thứ tự từng tầng (level order)**, `None` nghĩa là không có nút ở vị trí đó (giống cách LeetCode in cây).

**Ràng buộc:**
- Số nút mỗi cây trong khoảng `[1, 5000]`.
- `-10^9 <= Node.val, target <= 10^9`.

Trước hết ta tự dựng lớp `TreeNode` và hàm `build_tree` (biến list level-order thành cây) để chạy thử được.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Dựng cây từ list level-order kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def to_level_list(root):
    """Ngược lại: cây -> list level-order (bỏ None thừa ở cuối)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([2, 1, 4], [1, 0, 3], 5, True),
    ([0, -10, 10], [5, 1, 7, 0, 2], 18, False),
]
for a, b, t, want in examples:
    r1, r2 = build_tree(a), build_tree(b)
    print(to_level_list(r1), to_level_list(r2), "target =", t, "-> mong đợi", want)

[2, 1, 4] [1, 0, 3] target = 5 -> mong đợi True
[0, -10, 10] [5, 1, 7, 0, 2] target = 18 -> mong đợi False


## 3. Kiến thức nền

### 3.1 Cây nhị phân và lớp `TreeNode`

Một **cây nhị phân** (binary tree) gồm các nút; mỗi nút có giá trị `val` và tối đa hai con `left`, `right`. Nút không có cha là **gốc** (root). Trong Python ta biểu diễn "không có con" bằng `None`.

Ví dụ cây `[2,1,4]`: gốc là `2`, con trái `1`, con phải `4`.

In [2]:
r = build_tree([2, 1, 4])
print("gốc:", r.val, "| trái:", r.left.val, "| phải:", r.right.val)
print("con của nút 1:", r.left.left, r.left.right)

gốc: 2 | trái: 1 | phải: 4
con của nút 1: None None


### 3.2 Cây nhị phân tìm kiếm (BST) và tính chất inorder

**BST** là cây nhị phân thoả mãn: với mọi nút `x`, **mọi** giá trị trong cây con trái `< x.val` và **mọi** giá trị trong cây con phải `> x.val`.

**Duyệt trung thứ tự (inorder traversal)**: thăm cây con trái → thăm nút → thăm cây con phải. Với BST, inorder cho ra dãy **tăng dần**. Đây là "chìa khoá vàng" của rất nhiều bài BST.

Viết bản đệ quy trước vì dễ hiểu nhất:

In [3]:
def inorder_recursive(root, out):
    if root is None:
        return
    inorder_recursive(root.left, out)
    out.append(root.val)
    inorder_recursive(root.right, out)


vals = []
inorder_recursive(build_tree([5, 1, 7, 0, 2]), vals)
print(vals)  # tăng dần vì là BST
assert vals == sorted(vals)

[0, 1, 2, 5, 7]


### 3.3 Duyệt inorder không đệ quy bằng stack

Python mặc định chỉ cho đệ quy sâu khoảng `1000` tầng. Cây có `5000` nút có thể **lệch hẳn** (mọi nút chỉ có con phải — giống danh sách liên kết), khi đó độ sâu `= 5000` → `RecursionError`. Vì vậy ta học cách duyệt bằng **stack** (ngăn xếp, vào sau ra trước):

1. Đi trái hết cỡ, đẩy từng nút vào stack.
2. Lấy nút đỉnh stack ra → đó là nút nhỏ nhất chưa thăm → ghi nhận.
3. Chuyển sang cây con phải của nó và lặp lại bước 1.

In [4]:
def inorder_iterative(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:              # bước 1: đi trái hết cỡ
            stack.append(node)
            node = node.left
        node = stack.pop()       # bước 2: nút nhỏ nhất chưa thăm
        out.append(node.val)
        node = node.right        # bước 3: sang cây con phải
    return out


print(inorder_iterative(build_tree([5, 1, 7, 0, 2])))
print(inorder_iterative(build_tree([0, -10, 10])))

[0, 1, 2, 5, 7]
[-10, 0, 10]


### 3.4 Tìm kiếm trên BST

Muốn biết giá trị `v` có trong BST không: bắt đầu từ gốc; nếu `v == node.val` thì có; nếu `v < node.val` đi **trái**; ngược lại đi **phải**. Mỗi bước loại bỏ hẳn một cây con, nên chi phí `O(h)` với `h` là chiều cao cây (cây cân bằng `h ≈ log n`, cây lệch `h = n`).

In [5]:
def bst_contains(root, v):
    node = root
    while node:
        if v == node.val:
            return True
        node = node.left if v < node.val else node.right
    return False


r2 = build_tree([5, 1, 7, 0, 2])
print([(v, bst_contains(r2, v)) for v in [0, 2, 3, 7, 8]])

[(0, True), (2, True), (3, False), (7, True), (8, False)]


### 3.5 `set` — tập băm

`set` trong Python cho phép kiểm tra `x in s` trung bình `O(1)`. Đây là công cụ của lời giải Two Sum kinh điển: duyệt từng số `v`, hỏi xem `target - v` đã từng xuất hiện chưa.

In [6]:
s = {1, 2, 4}
target = 5
for v in [0, 1, 3]:
    print(v, "cần", target - v, "->", (target - v) in s)

0 cần 5 -> False
1 cần 4 -> True
3 cần 2 -> True


## 4. Ý tưởng nền tảng / trực giác

Bỏ hết chữ "cây" đi, bài toán chỉ còn là:

> Cho hai **tập số** `A` (giá trị cây 1) và `B` (giá trị cây 2). Có tồn tại `a ∈ A`, `b ∈ B` với `a + b = target` không?

Tính tay với ví dụ 1: `A = {1, 2, 4}`, `B = {0, 1, 3}`, `target = 5`.

| a | cần b = 5 − a | b có trong B? |
|---|---|---|
| 1 | 4 | không |
| 2 | 3 | **có** → `True` |
| 4 | 1 | có (cũng được) |

Ví dụ 2: `A = {-10, 0, 10}`, `B = {0, 1, 2, 5, 7}`, `target = 18`: cần `b ∈ {28, 18, 8}` — không cái nào có trong `B` → `False`.

Vậy câu hỏi thật sự là: **tận dụng tính chất BST thế nào** để làm nhanh hơn và/hoặc tốn ít bộ nhớ hơn?

In [7]:
A = inorder_iterative(build_tree([2, 1, 4]))
B = inorder_iterative(build_tree([1, 0, 3]))
print("A =", A, "B =", B)
print("các cặp có tổng 5:", [(a, b) for a in A for b in B if a + b == 5])

A = [1, 2, 4] B = [0, 1, 3]
các cặp có tổng 5: [(2, 3), (4, 1)]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** lấy toàn bộ giá trị hai cây ra hai list, thử **mọi cặp** `(a, b)`.

- Dễ viết, dễ đúng → dùng làm "trọng tài" để kiểm tra các cách nhanh hơn.
- Không dùng gì của BST cả.

In [8]:
def two_sum_bsts_brute(root1, root2, target):
    A = inorder_iterative(root1)
    B = inorder_iterative(root2)
    for a in A:
        for b in B:
            if a + b == target:
                return True
    return False


for a, b, t, want in examples:
    got = two_sum_bsts_brute(build_tree(a), build_tree(b), t)
    print(got, "mong đợi", want)
    assert got == want

True mong đợi True
False mong đợi False


**Độ phức tạp:** với `n`, `m` là số nút hai cây: Time `O(n·m)` (tệ nhất `5000 × 5000 = 25·10^6` phép cộng — chạy được nhưng chậm trong Python), Space `O(n + m)`.

**Nút thắt:** với mỗi `a`, ta quét **toàn bộ** `B` để tìm đúng một giá trị `target − a`. "Tìm một giá trị cụ thể" là việc mà `set` (O(1)) hoặc BST (O(h)) làm tốt hơn nhiều.

## 6. Bước đột phá

Có hai insight, mỗi cái dẫn tới một họ lời giải:

1. **"Tìm bạn cặp" là phép tra cứu**: với mỗi `a ∈ A`, ta chỉ cần biết `target − a` có trong `B` không. Tra cứu nhanh → dùng `set` hoặc tìm trên BST.
2. **Inorder cho dãy tăng dần**: có hai dãy **đã sắp xếp** `A` (tăng) và `B` (tăng) thì dùng **hai con trỏ**: `i` ở đầu `A` (nhỏ nhất), `j` ở cuối `B` (lớn nhất).
   - **Nếu** `A[i] + B[j] == target` **thì** tìm thấy.
   - **Nếu** tổng `< target` **thì** cần tổng lớn hơn → `i += 1` (B[j] đã là lớn nhất còn lại, giữ `i` thì không cặp nào với `A[i]` đủ lớn).
   - **Nếu** tổng `> target` **thì** cần tổng nhỏ hơn → `j -= 1`.

**Bất biến (invariant)** của hai con trỏ: mọi cặp có `A[i'] ` với `i' < i` hoặc `B[j']` với `j' > j` **đều đã được loại** một cách chắc chắn — không cặp nào trong đó cho tổng đúng `target`.

Insight 2 còn mở ra cách tiết kiệm bộ nhớ: không cần lưu cả dãy, chỉ cần một **iterator tăng dần** trên cây 1 và một **iterator giảm dần** trên cây 2 (mỗi cái dùng stack cỡ `O(h)`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hash set

Đổ toàn bộ giá trị cây 1 vào `set`, rồi duyệt cây 2: với mỗi `b`, nếu `target − b` thuộc `set` thì trả `True`.

In [9]:
def two_sum_bsts_set(root1, root2, target):
    seen = set(inorder_iterative(root1))
    for b in inorder_iterative(root2):
        if target - b in seen:
            return True
    return False


for a, b, t, want in examples:
    assert two_sum_bsts_set(build_tree(a), build_tree(b), t) == want
print("Cách 2 đúng trên ví dụ")

Cách 2 đúng trên ví dụ


**Độ phức tạp:** Time `O(n + m)` trung bình, Space `O(n)` cho `set` (cộng `O(m)` nếu ta lưu list cây 2; có thể duyệt cây 2 trực tiếp bằng stack để chỉ tốn `O(h2)`). Cách này **không dùng tính chất BST** — áp dụng được cho cả cây nhị phân thường.

### 7.2 Cách 3 — Duyệt cây 1, tìm kiếm nhị phân trên cây 2

Với mỗi giá trị `a` của cây 1, gọi `bst_contains(root2, target − a)`. Không cần bộ nhớ phụ ngoài stack duyệt.

In [10]:
def two_sum_bsts_search(root1, root2, target):
    stack, node = [], root1
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        if bst_contains(root2, target - node.val):
            return True
        node = node.right
    return False


for a, b, t, want in examples:
    assert two_sum_bsts_search(build_tree(a), build_tree(b), t) == want
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


**Độ phức tạp:** Time `O(n · h2)` — tốt (`O(n log m)`) nếu cây 2 cân bằng, nhưng tệ nhất `O(n·m)` nếu cây 2 lệch. Space `O(h1)`.

### 7.3 Cách 4 — Inorder thành hai dãy + hai con trỏ

Đây là lời giải "chuẩn" (cũng là lời giải tham khảo trên doocs): inorder hai cây thành `A`, `B` tăng dần, rồi `i = 0`, `j = len(B) − 1`.

In [11]:
def two_sum_bsts_two_pointers(root1, root2, target):
    A = inorder_iterative(root1)
    B = inorder_iterative(root2)
    i, j = 0, len(B) - 1
    while i < len(A) and j >= 0:
        s = A[i] + B[j]
        if s == target:
            return True
        if s < target:
            i += 1
        else:
            j -= 1
    return False


for a, b, t, want in examples:
    assert two_sum_bsts_two_pointers(build_tree(a), build_tree(b), t) == want
print("Cách 4 đúng trên ví dụ")

Cách 4 đúng trên ví dụ


**Độ phức tạp:** Time `O(n + m)` (mỗi bước tăng `i` hoặc giảm `j`, tổng tối đa `n + m` bước), Space `O(n + m)` cho hai dãy. Không cần băm → thời gian ổn định, không phụ thuộc va chạm hash.

### 7.4 Cách 5 — Hai iterator bằng stack (bộ nhớ O(h1 + h2))

Thay vì lưu cả dãy, ta làm hai "máy phát":
- `asc(root1)`: trả lần lượt các giá trị **tăng dần** (inorder trái → nút → phải).
- `desc(root2)`: trả lần lượt các giá trị **giảm dần** (inorder ngược: phải → nút → trái).

Mỗi máy phát chỉ giữ stack các nút "đang dang dở" — tối đa bằng chiều cao cây. Trong Python, cách gọn nhất là dùng **generator** (hàm có `yield`).

In [12]:
def asc(root):
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        yield node.val
        node = node.right


def desc(root):
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.right
        node = stack.pop()
        yield node.val
        node = node.left


r = build_tree([5, 1, 7, 0, 2])
print("tăng :", list(asc(r)))
print("giảm :", list(desc(r)))

tăng : [0, 1, 2, 5, 7]
giảm : [7, 5, 2, 1, 0]


In [13]:
def two_sum_bsts_iter(root1, root2, target):
    it1, it2 = asc(root1), desc(root2)
    a = next(it1, None)
    b = next(it2, None)
    while a is not None and b is not None:
        s = a + b
        if s == target:
            return True
        if s < target:
            a = next(it1, None)
        else:
            b = next(it2, None)
    return False


for a, b, t, want in examples:
    assert two_sum_bsts_iter(build_tree(a), build_tree(b), t) == want
print("Cách 5 đúng trên ví dụ")

Cách 5 đúng trên ví dụ


**Độ phức tạp:** Time `O(n + m)` (mỗi nút được đẩy/lấy khỏi stack đúng một lần), Space `O(h1 + h2)`. Với cây cân bằng đó là `O(log n + log m)` — rất tiết kiệm. Lưu ý dùng `next(it, None)` an toàn vì giá trị nút là số nguyên, không bao giờ là `None`.

## 8. TRACE từng bước

Trace Cách 4 (hai con trỏ) trên ví dụ 1: `A = [1, 2, 4]`, `B = [0, 1, 3]`, `target = 5`.

| Bước | i | A[i] | j | B[j] | Tổng | So với 5 | Hành động |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 1 | 2 | 3 | 4 | nhỏ hơn | `i += 1` |
| 2 | 1 | 2 | 2 | 3 | 5 | bằng | trả `True` |

Và ví dụ 2 (`A = [-10, 0, 10]`, `B = [0, 1, 2, 5, 7]`, `target = 18`): tổng luôn nhỏ hơn 18 nên `i` tăng đến hết → `False`. Hàm `trace_two_pointers` in đúng từng trạng thái:

In [14]:
def trace_two_pointers(root1, root2, target):
    A, B = inorder_iterative(root1), inorder_iterative(root2)
    print("A =", A, "| B =", B, "| target =", target)
    i, j, step = 0, len(B) - 1, 1
    while i < len(A) and j >= 0:
        s = A[i] + B[j]
        if s == target:
            print(f"bước {step}: i={i} A[i]={A[i]:>3} j={j} B[j]={B[j]:>3} tổng={s:>3} == target -> True")
            return True
        act = "i += 1" if s < target else "j -= 1"
        print(f"bước {step}: i={i} A[i]={A[i]:>3} j={j} B[j]={B[j]:>3} tổng={s:>3} {'<' if s < target else '>'} target -> {act}")
        if s < target:
            i += 1
        else:
            j -= 1
        step += 1
    print("con trỏ chạy ra ngoài -> False")
    return False


for a, b, t, _ in examples:
    trace_two_pointers(build_tree(a), build_tree(b), t)
    print()

A = [1, 2, 4] | B = [0, 1, 3] | target = 5
bước 1: i=0 A[i]=  1 j=2 B[j]=  3 tổng=  4 < target -> i += 1
bước 2: i=1 A[i]=  2 j=2 B[j]=  3 tổng=  5 == target -> True

A = [-10, 0, 10] | B = [0, 1, 2, 5, 7] | target = 18
bước 1: i=0 A[i]=-10 j=4 B[j]=  7 tổng= -3 < target -> i += 1
bước 2: i=1 A[i]=  0 j=4 B[j]=  7 tổng=  7 < target -> i += 1
bước 3: i=2 A[i]= 10 j=4 B[j]=  7 tổng= 17 < target -> i += 1
con trỏ chạy ra ngoài -> False



Trace thêm trạng thái **stack** của Cách 5 để thấy bộ nhớ chỉ bằng chiều cao cây. Hàm `trace_asc` in stack sau mỗi lần lấy ra một giá trị:

In [15]:
def trace_asc(root):
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        print(f"lấy {node.val:>3} | stack còn: {[x.val for x in stack]}")
        node = node.right


trace_asc(build_tree([5, 1, 7, 0, 2]))

lấy   0 | stack còn: [5, 1]
lấy   1 | stack còn: [5]
lấy   2 | stack còn: [5]
lấy   5 | stack còn: []
lấy   7 | stack còn: []


## 9. Code nộp LeetCode

Chọn Cách 5 (hai iterator, `O(n+m)` thời gian, `O(h1+h2)` bộ nhớ, hoàn toàn không đệ quy nên an toàn với cây lệch 5000 nút). Trên LeetCode đã có sẵn `TreeNode` và `Optional`; ở đây ta đã định nghĩa ở trên.

In [16]:
class Solution:
    def twoSumBSTs(self, root1: Optional[TreeNode], root2: Optional[TreeNode], target: int) -> bool:
        def asc(root):
            stack, node = [], root
            while stack or node:
                while node:
                    stack.append(node)
                    node = node.left
                node = stack.pop()
                yield node.val
                node = node.right

        def desc(root):
            stack, node = [], root
            while stack or node:
                while node:
                    stack.append(node)
                    node = node.right
                node = stack.pop()
                yield node.val
                node = node.left

        it1, it2 = asc(root1), desc(root2)
        a, b = next(it1, None), next(it2, None)
        while a is not None and b is not None:
            s = a + b
            if s == target:
                return True
            if s < target:
                a = next(it1, None)
            else:
                b = next(it2, None)
        return False


sol = Solution()
for a, b, t, want in examples:
    got = sol.twoSumBSTs(build_tree(a), build_tree(b), t)
    print(got)
    assert got == want

True
False


## 10. Vì sao đúng

Gọi `A` là dãy tăng dần (cây 1), `B` là dãy tăng dần (cây 2). Thuật toán duy trì **bất biến**:

> Nếu tồn tại cặp `(p, q)` với `A[p] + B[q] = target`, thì luôn có `p ≥ i` và `q ≤ j`.

- Ban đầu `i = 0`, `j = len(B)−1` → hiển nhiên đúng.
- Khi `A[i] + B[j] < target`: với mọi `q ≤ j` ta có `B[q] ≤ B[j]`, nên `A[i] + B[q] < target`. Vậy `A[i]` **không thể** nằm trong cặp đáp án nào còn lại → bỏ `i` an toàn (`i += 1`).
- Khi `A[i] + B[j] > target`: đối xứng, với mọi `p ≥ i`, `A[p] + B[j] ≥ A[i] + B[j] > target` → bỏ `j` an toàn.
- Mỗi bước thu hẹp vùng tìm kiếm ít nhất một phần tử, nên thuật toán dừng. Nếu có cặp đáp án, vùng `[i..] × [..j]` luôn chứa nó, nên ta sẽ gặp đúng lúc `A[i] + B[j] = target` trước khi một con trỏ chạy ra ngoài.

Cách 5 chỉ là Cách 4 trong đó `A[i]` được sinh "theo yêu cầu" bởi `asc` và `B[j]` bởi `desc` — thứ tự giá trị y hệt, nên cùng đúng.

Vì đáp án yêu cầu một nút ở **mỗi** cây, ta lấy `a` luôn từ `A` và `b` luôn từ `B` — không bao giờ ghép hai nút cùng cây.

## 11. Time / Space Complexity

Gọi `n`, `m` là số nút của cây 1, cây 2; `h1`, `h2` là chiều cao.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n·m)` | `O(n + m)` |
| 2. Hash set | `O(n + m)` trung bình | `O(n)` |
| 3. Tìm trên BST 2 | `O(n·h2)` (cân bằng: `O(n log m)`, lệch: `O(n·m)`) | `O(h1)` |
| 4. Hai dãy + hai con trỏ | `O(n + m)` | `O(n + m)` |
| 5. Hai iterator | `O(n + m)` | `O(h1 + h2)` (tệ nhất `O(n + m)` khi cây lệch) |

Với giới hạn `n, m ≤ 5000`, các cách `O(n + m)` chạy cỡ vài mili-giây.

## 12. Bộ test

### 12.1 Ví dụ của đề + edge case

In [17]:
methods = {
    "brute": two_sum_bsts_brute,
    "set": two_sum_bsts_set,
    "search": two_sum_bsts_search,
    "two_ptr": two_sum_bsts_two_pointers,
    "iter": two_sum_bsts_iter,
    "submit": lambda r1, r2, t: Solution().twoSumBSTs(r1, r2, t),
}

edge_cases = [
    ([1], [2], 3, True),             # mỗi cây 1 nút, đúng
    ([1], [2], 4, False),            # mỗi cây 1 nút, sai
    ([10 ** 9], [10 ** 9], 2 * 10 ** 9, True),   # giá trị lớn (Python không tràn số)
    ([-10 ** 9], [-10 ** 9], -10 ** 9, False),
    ([1], [10], 2, False),           # 1+1=2 nhưng cả hai số 1 phải từ cùng cây 1 -> sai
    ([5, 3, 8], [5, 3, 8], 16, True),  # 8 + 8 từ hai cây khác nhau -> hợp lệ
]
for name, f in methods.items():
    for a, b, t, want in examples + edge_cases:
        assert f(build_tree(a), build_tree(b), t) == want, (name, a, b, t)
print("Tất cả cách đúng trên ví dụ + edge case")

Tất cả cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh BST ngẫu nhiên bằng cách **chèn** các giá trị khác nhau theo thứ tự ngẫu nhiên (hàm `insert_bst` dạng lặp).

In [18]:
def insert_bst(root, v):
    if root is None:
        return TreeNode(v)
    node = root
    while True:
        if v < node.val:
            if node.left is None:
                node.left = TreeNode(v)
                return root
            node = node.left
        else:
            if node.right is None:
                node.right = TreeNode(v)
                return root
            node = node.right


def random_bst(rng, size, lo, hi):
    vals = rng.sample(range(lo, hi + 1), size)
    root = None
    for v in vals:
        root = insert_bst(root, v)
    return root


rng = random.Random(1214)
for _ in range(400):
    r1 = random_bst(rng, rng.randint(1, 12), -20, 20)
    r2 = random_bst(rng, rng.randint(1, 12), -20, 20)
    t = rng.randint(-40, 40)
    want = two_sum_bsts_brute(r1, r2, t)
    for name, f in methods.items():
        assert f(r1, r2, t) == want, name
print("400 test ngẫu nhiên: mọi cách khớp brute force")

400 test ngẫu nhiên: mọi cách khớp brute force


### 12.3 Test giới hạn: cây lệch 5000 nút

Cây lệch (mọi nút chỉ có con phải) là ca khó nhất cho đệ quy và cho Cách 3. Ta dựng bằng vòng lặp.

In [19]:
def chain_right(values):
    """values tăng dần -> cây lệch phải."""
    root = TreeNode(values[0])
    node = root
    for v in values[1:]:
        node.right = TreeNode(v)
        node = node.right
    return root


def balanced_from_sorted(values):
    """Dựng BST cân bằng từ dãy tăng (đệ quy chỉ sâu ~log n)."""
    def go(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(values[mid], go(lo, mid - 1), go(mid + 1, hi))
    return go(0, len(values) - 1)


big1 = chain_right(list(range(0, 10000, 2)))         # 5000 số chẵn
big2 = balanced_from_sorted(list(range(1, 10001, 2)))  # 5000 số lẻ
for name in ["set", "two_ptr", "iter", "submit"]:
    t0 = time.perf_counter()
    r_true = methods[name](big1, big2, 9999 + 9998)     # 9998 + 9999
    r_false = methods[name](big1, big2, 4)              # chẵn + lẻ luôn lẻ -> False
    dt = time.perf_counter() - t0
    assert r_true is True and r_false is False
    print(f"{name:8s} OK trong {dt * 1000:.1f} ms")

set      OK trong 2.3 ms
two_ptr  OK trong 2.5 ms
iter     OK trong 1.3 ms
submit   OK trong 1.3 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gộp giá trị hai cây vào một tập rồi làm Two Sum

Code sai: đổ cả hai cây vào một `set`, rồi tìm `target − v`.

In [20]:
def wrong_merge(root1, root2, target):
    allv = set(inorder_iterative(root1)) | set(inorder_iterative(root2))
    return any(target - v in allv for v in allv)


r1, r2 = build_tree([1]), build_tree([10])
print("sai :", wrong_merge(r1, r2, 2))          # True — ghép 1 với chính nó!
print("đúng:", two_sum_bsts_set(r1, r2, 2))     # False

sai : True
đúng: False


**Vì sao sai:** đề bắt **một nút ở mỗi cây**. Gộp lại thì có thể ghép hai số của cùng một cây (thậm chí cùng một nút với chính nó). **Sửa:** `set` chỉ chứa cây 1, và chỉ duyệt cây 2 để tra (Cách 2).

### Lỗi 2 — Hai con trỏ cùng xuất phát từ đầu hai dãy

Code sai: `i = 0`, `j = 0`, tổng nhỏ thì tăng... cái nào?

In [21]:
def wrong_same_direction(root1, root2, target):
    A, B = inorder_iterative(root1), inorder_iterative(root2)
    i = j = 0
    while i < len(A) and j < len(B):
        s = A[i] + B[j]
        if s == target:
            return True
        if s < target:
            i += 1          # tại sao không phải j += 1? Không có lý do chính đáng!
        else:
            return False    # tổng đã lớn hơn và chỉ có thể tăng
    return False


r1, r2 = build_tree([2, 1, 4]), build_tree([1, 0, 3])
print("sai :", wrong_same_direction(r1, r2, 5))     # False
print("đúng:", two_sum_bsts_two_pointers(r1, r2, 5))  # True (2 + 3)

sai : False
đúng: True


**Vì sao sai:** khi cả hai con trỏ cùng ở đầu nhỏ, cả hai hướng di chuyển đều **làm tổng tăng** — không có quy tắc nào để biết nên tăng con trỏ nào mà không bỏ sót. Ở đây `i` chạy hết `A` trong khi `j` mãi ở `B[0] = 0`, bỏ lỡ cặp `(2, 3)`. **Sửa:** một con trỏ ở đầu nhỏ của `A`, một con trỏ ở đầu **lớn** của `B` — khi đó hai hướng di chuyển có tác dụng **ngược nhau** (tăng / giảm tổng).

### Lỗi 3 — Tìm trong BST nhưng đi sai hướng

In [22]:
def wrong_contains(root, v):
    node = root
    while node:
        if v == node.val:
            return True
        node = node.right if v < node.val else node.left   # đảo ngược!
    return False


r2 = build_tree([5, 1, 7, 0, 2])
print("sai :", [wrong_contains(r2, v) for v in [0, 2, 7]])
print("đúng:", [bst_contains(r2, v) for v in [0, 2, 7]])

sai : [False, False, False]
đúng: [True, True, True]


**Vì sao sai:** giá trị nhỏ hơn nút nằm ở **trái**. Đi nhầm sang phải thì không bao giờ gặp. Mẹo nhớ: trên trục số, "nhỏ" nằm bên **trái**. **Sửa:** `node.left if v < node.val else node.right`.

### Lỗi 4 — Duyệt đệ quy trên cây lệch 5000 nút

In [23]:
deep = chain_right(list(range(5000)))
try:
    out = []
    inorder_recursive(deep, out)
    print("đệ quy chạy được, độ dài", len(out))
except RecursionError as e:
    print("đệ quy lỗi:", type(e).__name__)
print("bản lặp:", len(inorder_iterative(deep)), "giá trị")

đệ quy lỗi: RecursionError
bản lặp: 5000 giá trị


**Vì sao sai:** giới hạn đệ quy mặc định của Python (~1000) nhỏ hơn độ sâu `5000`. Trên LeetCode giới hạn có thể được nới, nhưng đừng trông chờ điều đó. **Sửa:** dùng stack tường minh (mục 3.3) hoặc `sys.setrecursionlimit` (kém an toàn hơn).

## 14. Mẹo & nhận dạng pattern

- Thấy **BST** → nghĩ ngay "inorder = dãy tăng dần". Nhiều bài BST thực chất là bài **mảng đã sắp xếp** đội lốt cây.
- Thấy "**tổng hai phần tử bằng target**" → Two Sum: `set` (dữ liệu bất kỳ) hoặc **hai con trỏ** (dữ liệu đã sắp xếp).
- Hai nguồn dữ liệu độc lập (hai cây, hai mảng) → con trỏ **ngược chiều**: nhỏ nhất của nguồn này, lớn nhất của nguồn kia.
- Muốn tiết kiệm bộ nhớ trên cây → **iterator bằng stack** (`BSTIterator`, bài 173). Bản đảo ngược (phải → nút → trái) cho dãy giảm dần.
- Cây có thể sâu đến hàng nghìn → ưu tiên duyệt **lặp**.

## 15. Biến thể

1. **Two Sum IV — Input is a BST (LeetCode 653):** chỉ một cây, tìm hai nút **khác nhau** có tổng `target`. Dùng `asc` và `desc` trên **cùng** cây, dừng khi hai iterator gặp nhau.
2. **Đếm số cặp** `(a ∈ cây 1, b ∈ cây 2)` có tổng `target` — với giá trị phân biệt trong mỗi cây, hai con trỏ khi gặp tổng bằng thì tăng `i` **và** giảm `j`.
3. **Cặp có tổng gần target nhất** — hai con trỏ, cập nhật `|A[i] + B[j] − target|` nhỏ nhất.
4. **Ba cây** — cố định một giá trị cây 1, chạy hai con trỏ trên cây 2 và cây 3: `O(n·(m + k))`.
5. Cây là **cây nhị phân thường** (không phải BST) → mất tính chất sắp xếp, dùng Cách 2 (hash set).

In [24]:
def two_sum_one_bst(root, target):
    """Biến thể 1 (LC 653): hai nút khác nhau trong cùng một BST."""
    it1, it2 = asc(root), desc(root)
    a, b = next(it1), next(it2)
    while a < b:                     # a < b đảm bảo hai nút khác nhau (giá trị phân biệt)
        s = a + b
        if s == target:
            return True
        if s < target:
            a = next(it1)
        else:
            b = next(it2)
    return False


r = build_tree([5, 3, 6, 2, 4, None, 7])
print(two_sum_one_bst(r, 9), two_sum_one_bst(r, 28), two_sum_one_bst(r, 10))

True False True


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi cặp | `O(n·m)` | `O(n+m)` | làm trọng tài kiểm tra |
| 2. Hash set | tra `target − b` trong set cây 1 | `O(n+m)` TB | `O(n)` | cây thường, viết nhanh nhất |
| 3. Tìm trên BST | mỗi `a` tìm `target − a` trong cây 2 | `O(n·h2)` | `O(h1)` | cây 2 cân bằng, không muốn tốn bộ nhớ |
| 4. Hai con trỏ trên list | inorder → 2 dãy tăng | `O(n+m)` | `O(n+m)` | dễ giải thích, ổn định |
| 5. Hai iterator | asc(cây 1) + desc(cây 2) | `O(n+m)` | `O(h1+h2)` | phỏng vấn hỏi tối ưu bộ nhớ |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Không chạy, đoán xem `two_sum_bsts_two_pointers(build_tree([4, 2, 6]), build_tree([3, 1, 5]), 7)` trả gì, và con trỏ đi qua những cặp nào?

### Bài 2 (tính tay)

Với `A = [-3, 0, 4, 9]`, `B = [-5, 2, 6]`, `target = 1`: lập bảng `i, j, A[i], B[j], tổng, hành động` như mục 8.

### Bài 3 (điền TODO) — đếm số cặp

Viết `count_pairs(root1, root2, target)` đếm số cặp `(a ∈ cây 1, b ∈ cây 2)` có `a + b == target` (giá trị trong mỗi cây phân biệt).

In [25]:
def count_pairs(root1, root2, target):
    A, B = inorder_iterative(root1), inorder_iterative(root2)
    i, j, cnt = 0, len(B) - 1, 0
    # TODO: chạy hai con trỏ; khi tổng bằng target thì cnt += 1 và di chuyển CẢ HAI con trỏ
    raise NotImplementedError

# print(count_pairs(build_tree([2, 1, 4]), build_tree([1, 0, 3]), 5))  # mong đợi 2: (2,3) và (4,1)

### Bài 4 (tự viết lại) — lớp iterator

Viết lại lớp `BSTIterator(root, reverse=False)` với hai method `has_next()` và `next()` (không dùng `yield`), rồi dùng nó viết lại Cách 5.

In [26]:
class BSTIteratorTodo:
    def __init__(self, root, reverse=False):
        # TODO: lưu stack, hướng đi; đẩy "nhánh trái" (hoặc phải nếu reverse) ban đầu
        pass

    def has_next(self):
        pass

    def next(self):
        pass

# it = BSTIteratorTodo(build_tree([5, 1, 7, 0, 2]), reverse=True)
# out = []
# while it.has_next(): out.append(it.next())
# print(out)   # mong đợi [7, 5, 2, 1, 0]

### Bài 5 (mini challenge) — tổng gần target nhất

Viết `closest_sum(root1, root2, target)` trả về giá trị `a + b` (một nút mỗi cây) **gần** `target` nhất; nếu hoà, chọn tổng nhỏ hơn.

In [27]:
def closest_sum_todo(root1, root2, target):
    # TODO: hai con trỏ, cập nhật best theo khoảng cách |s - target|
    raise NotImplementedError

# print(closest_sum_todo(build_tree([2, 1, 4]), build_tree([1, 0, 3]), 10))  # mong đợi 7 (4 + 3)

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: inorder hai cây trước. Mức 2: `A = [2,4,6]`, `B = [1,3,5]`. Mức 3: bắt đầu `(2,5)` tổng 7 → dừng ngay.

**Bài 2.** Mức 1: `i = 0`, `j = 2`. Mức 2: bước đầu `-3 + 6 = 3 > 1` → `j -= 1`. Mức 3: tiếp `-3 + 2 = -1 < 1` → `i += 1`, rồi `0 + 2 = 2 > 1` → `j -= 1`, `0 + (-5) < 1` → `i += 1`, `4 + (-5) = -1` → `i += 1`, `9 − 5 = 4 > 1` → `j = -1` → dừng, `False`.

**Bài 3.** Mức 1: giữ khung hai con trỏ. Mức 2: khi bằng, `A[i]` không thể ghép với `B` khác (giá trị phân biệt) nên `i += 1`; tương tự `j -= 1`. Mức 3: xem đáp án mục 19.

**Bài 4.** Mức 1: hàm phụ `_push(node)` đẩy cả nhánh. Mức 2: `next()` = `pop` rồi `_push(con bên kia)`. Mức 3: `has_next()` = `len(stack) > 0`.

**Bài 5.** Mức 1: khung giống Cách 4. Mức 2: so sánh bộ `(abs(s − target), s)` để xử lý hoà. Mức 3: nếu `s == target` có thể trả ngay.

## 19. Đáp án tham khảo

**Bài 1:** `True` — cặp đầu tiên `(2, 5)` đã có tổng 7.

**Bài 2:** kết quả `False` (các bước như gợi ý mức 3).

In [28]:
print(two_sum_bsts_two_pointers(build_tree([4, 2, 6]), build_tree([3, 1, 5]), 7))
assert two_sum_bsts_two_pointers(build_tree([4, 2, 6]), build_tree([3, 1, 5]), 7) is True
trace_two_pointers(balanced_from_sorted([-3, 0, 4, 9]), balanced_from_sorted([-5, 2, 6]), 1)

True
A = [-3, 0, 4, 9] | B = [-5, 2, 6] | target = 1
bước 1: i=0 A[i]= -3 j=2 B[j]=  6 tổng=  3 > target -> j -= 1
bước 2: i=0 A[i]= -3 j=1 B[j]=  2 tổng= -1 < target -> i += 1
bước 3: i=1 A[i]=  0 j=1 B[j]=  2 tổng=  2 > target -> j -= 1
bước 4: i=1 A[i]=  0 j=0 B[j]= -5 tổng= -5 < target -> i += 1
bước 5: i=2 A[i]=  4 j=0 B[j]= -5 tổng= -1 < target -> i += 1
bước 6: i=3 A[i]=  9 j=0 B[j]= -5 tổng=  4 > target -> j -= 1
con trỏ chạy ra ngoài -> False


False

In [29]:
def count_pairs(root1, root2, target):
    A, B = inorder_iterative(root1), inorder_iterative(root2)
    i, j, cnt = 0, len(B) - 1, 0
    while i < len(A) and j >= 0:
        s = A[i] + B[j]
        if s == target:
            cnt += 1
            i += 1
            j -= 1
        elif s < target:
            i += 1
        else:
            j -= 1
    return cnt


class BSTIterator:
    def __init__(self, root, reverse=False):
        self.stack, self.reverse = [], reverse
        self._push(root)

    def _push(self, node):
        while node:
            self.stack.append(node)
            node = node.right if self.reverse else node.left

    def has_next(self):
        return len(self.stack) > 0

    def next(self):
        node = self.stack.pop()
        self._push(node.left if self.reverse else node.right)
        return node.val


def closest_sum(root1, root2, target):
    A, B = inorder_iterative(root1), inorder_iterative(root2)
    i, j = 0, len(B) - 1
    best = None
    while i < len(A) and j >= 0:
        s = A[i] + B[j]
        if best is None or (abs(s - target), s) < (abs(best - target), best):
            best = s
        if s == target:
            return s
        if s < target:
            i += 1
        else:
            j -= 1
    return best


assert count_pairs(build_tree([2, 1, 4]), build_tree([1, 0, 3]), 5) == 2
it = BSTIterator(build_tree([5, 1, 7, 0, 2]), reverse=True)
out = []
while it.has_next():
    out.append(it.next())
assert out == [7, 5, 2, 1, 0]
assert closest_sum(build_tree([2, 1, 4]), build_tree([1, 0, 3]), 10) == 7

# kiểm tra ngẫu nhiên count_pairs và closest_sum với vét cạn
rng = random.Random(7)
for _ in range(200):
    r1 = random_bst(rng, rng.randint(1, 8), -10, 10)
    r2 = random_bst(rng, rng.randint(1, 8), -10, 10)
    t = rng.randint(-20, 20)
    A, B = inorder_iterative(r1), inorder_iterative(r2)
    assert count_pairs(r1, r2, t) == sum(a + b == t for a in A for b in B)
    assert closest_sum(r1, r2, t) == min((a + b for a in A for b in B), key=lambda s: (abs(s - t), s))
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
def asc(root):                      # inorder tăng dần, bộ nhớ O(h)
    stack, node = [], root
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop(); yield node.val
        node = node.right
# desc: đổi left <-> right

a, b = next(it1, None), next(it2, None)
while a is not None and b is not None:
    s = a + b
    if s == target: return True
    if s < target: a = next(it1, None)
    else:          b = next(it2, None)
return False
```

**Checklist:**
- [ ] Hai số lấy từ **hai cây khác nhau** (không gộp chung)?
- [ ] Con trỏ **ngược chiều**: tăng dần ở cây 1, giảm dần ở cây 2?
- [ ] Duyệt **không đệ quy** (cây lệch 5000 nút)?
- [ ] `next(it, None)` để không văng `StopIteration`?
- [ ] Giá trị âm và lớn đến `10^9` — Python không tràn số, nhưng nếu viết C++/Java nhớ dùng `long`?
- [ ] Đã thử: ví dụ đề, mỗi cây 1 nút, cặp nằm ở hai đầu, không có cặp nào?